# Scenario: You need to create a binary tree from given input data, such as an array, inorder and preorder/postorder traversal sequences, or level order traversal.

# Clue: Explore problems where you must build a binary tree based on provided specifications, reconstructing the tree from its serialized form or given traversal sequences.

1. **https://leetcode.com/problems/construct-binary-tree-from-preorder-and-inorder-traversal/**

In [ ]:
# brute solution -> recursion/divide and conquer
# time -> O(n2), as each call does n works, so n + n-1 + n-2....1 = n2
# space -> O(n), maximum memory alive at one moment → O(n)


# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        if len(preorder) == 0:
            return None
        root_val = preorder[0]
        root = TreeNode(root_val)
        indx = inorder.index(root_val)

        root.left = self.buildTree(preorder[1:indx+1],inorder[:indx])
        root.right = self.buildTree(preorder[indx+1:],inorder[indx+1:])
        return root

In [ ]:
# optimal hashmap
# time = O(n)
# space = O(n)

# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def buildTree(self, preorder: List[int], inorder: List[int]) -> Optional[TreeNode]:
        preorder_indx = 0
        inorder_indx = {}
        for i in range(len(inorder)):
            inorder_indx[inorder[i]] = i
        def build(left,right):
            nonlocal preorder_indx
            if(left > right):
                return None
            root_val = preorder[preorder_indx]
            preorder_indx += 1
            root = TreeNode(root_val)
            mid = inorder_indx[root_val]
            root.left = build(left,mid-1)
            root.right = build(mid+1,right)
            return root

        return build(0,len(inorder)-1)

2. **https://leetcode.com/problems/construct-binary-tree-from-inorder-and-postorder-traversal/description/**

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def buildTree(self, inorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        postorder_indx = len(postorder)-1
        inorder_indx = {}
        for i in range(len(inorder)):
            inorder_indx[inorder[i]] = i
        
        def build(left,right):
            nonlocal postorder_indx
            if(left > right):
                return None
            root_val = postorder[postorder_indx]
            postorder_indx -= 1
            root = TreeNode(root_val)
            mid = inorder_indx[root_val]
            root.right = build(mid+1,right)
            root.left = build(left,mid-1)
            return root
        return build(0,len(inorder)-1)

3. **https://leetcode.com/problems/construct-binary-tree-from-preorder-and-postorder-traversal/description/**

In [ ]:
# brute force

class Solution:
    def constructFromPrePost(self, preorder, postorder):
        # No nodes
        if not preorder:
            return None
        # First preorder value is root
        root = TreeNode(preorder[0])
        # Only one node
        if len(preorder) == 1:
            return root
        # Next preorder value is the root of left subtree
        left_root = preorder[1]
        # Find where left subtree ends in postorder
        index = postorder.index(left_root)
        # Number of nodes in left subtree
        left_size = index + 1
        root.left = self.constructFromPrePost(
            preorder[1 : left_size + 1],
            postorder[:left_size]
        )
        root.right = self.constructFromPrePost(
            preorder[left_size + 1 :],
            postorder[left_size : -1]
        )
        return root

In [ ]:
class Solution:
    def constructFromPrePost(
        self,
        preorder: List[int],
        postorder: List[int]
    ) -> Optional[TreeNode]:
        # value -> index in original postorder
        post_index = {}
        for i in range(len(postorder)):
            post_index[postorder[i]] = i
        def build(pre_start, pre_end, post_start, post_end):
            # No nodes
            if pre_start > pre_end:
                return None
            # First preorder value is root
            root_val = preorder[pre_start]
            root = TreeNode(root_val)
            # Only one node
            if pre_start == pre_end:
                return root
            # Next preorder value is root of left subtree
            left_root = preorder[pre_start + 1]
            # Global position of left root in postorder
            left_root_index = post_index[left_root]
            # Number of nodes in left subtree
            left_size = left_root_index - post_start + 1
            # Build left subtree
            root.left = build(
                pre_start + 1,
                pre_start + left_size,
                post_start,
                left_root_index
            )
            # Build right subtree
            root.right = build(
                pre_start + left_size + 1,
                pre_end,
                left_root_index + 1,
                post_end - 1
            )
            return root
        n = len(preorder)
        return build(
            0, n - 1,
            0, n - 1
        )

For this solution, **do not dry-run all four arrays or redraw the tree after every call**. That creates unnecessary cognitive load.

Use one compact table with **one row per `build()` call**.

Your function is:

```python
build(pre_start, pre_end, post_start, post_end)
```

For each call, write only:

```text
Call | Root | LeftRoot | LeftRootPostIdx | LeftSize | Left Call | Right Call
```

That's enough.

For:

```text
preorder  = [1,2,4,5,3,6,7]
postorder = [4,5,2,6,7,3,1]
```

First write the postorder index map in one line:

```text
post_index:
4:0  5:1  2:2  6:3  7:4  3:5  1:6
```

Then start.

### Call 1

```text
build(0,6,0,6)

root      = preorder[0] = 1
left_root = preorder[1] = 2

left_root_index = post_index[2] = 2

left_size = 2 - 0 + 1 = 3
```

Therefore:

```text
LEFT  = build(1,3,0,2)
RIGHT = build(4,6,3,5)
```

On paper, compress all of that to:

```text
build(0,6,0,6)
root=1, LR=2, idx=2, size=3
L → (1,3,0,2)
R → (4,6,3,5)
```

That's the format I recommend.

---

Now recursively do left:

```text
build(1,3,0,2)
```

Calculate:

```text
root      = preorder[1] = 2
left_root = preorder[2] = 4

index = post_index[4] = 0

left_size = 0 - 0 + 1 = 1
```

Children:

```text
L → build(2,2,0,0)
R → build(3,3,1,1)
```

Write:

```text
build(1,3,0,2)
root=2, LR=4, idx=0, size=1
L → (2,2,0,0)
R → (3,3,1,1)
```

Now:

```text
build(2,2,0,0)
```

Since:

```text
pre_start == pre_end
```

single node:

```text
root = preorder[2] = 4
return 4
```

Just write:

```text
(2,2,0,0) → 4
```

Next:

```text
(3,3,1,1) → 5
```

So now mentally:

```text
    2
   / \
  4   5
```

---

Then process the right subtree from Call 1:

```text
build(4,6,3,5)

root      = preorder[4] = 3
left_root = preorder[5] = 6

index = post_index[6] = 3

left_size = 3 - 3 + 1 = 1
```

So:

```text
L → (5,5,3,3)
R → (6,6,4,4)
```

Both are single nodes:

```text
(5,5,3,3) → 6
(6,6,4,4) → 7
```

Giving:

```text
    3
   / \
  6   7
```

Final tree:

```text
        1
      /   \
     2     3
    / \   / \
   4   5 6   7
```

## What your interview paper should actually look like

Don't write everything I wrote above. Write only this:

```text
postIdx: 4:0 5:1 2:2 6:3 7:4 3:5 1:6


(0,6,0,6)
root=1 LR=2 idx=2 size=3
L(1,3,0,2)    R(4,6,3,5)


(1,3,0,2)
root=2 LR=4 idx=0 size=1
L(2,2,0,0)    R(3,3,1,1)

(2,2,0,0) → 4
(3,3,1,1) → 5


(4,6,3,5)
root=3 LR=6 idx=3 size=1
L(5,5,3,3)    R(6,6,4,4)

(5,5,3,3) → 6
(6,6,4,4) → 7
```

That's it.

### The 4-step routine to repeat

For every non-leaf call, calculate only:

```text
1. root      = preorder[pre_start]

2. leftRoot  = preorder[pre_start + 1]

3. idx       = post_index[leftRoot]

4. leftSize  = idx - post_start + 1
```

Then derive:

```text
LEFT:
pre  = pre_start+1 → pre_start+leftSize
post = post_start   → idx

RIGHT:
pre  = pre_start+leftSize+1 → pre_end
post = idx+1                 → post_end-1
```

The key is: **don't dry-run the code line by line**. Dry-run the **recursive calls/ranges**. That's what matters in an interview.


In [ ]:
# below solution is only for better understanding of recursion

# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:            
        def makeTree():            
            node = TreeNode(postorder.pop())

            if node.val != preorder[-1]:
                node.right = makeTree()

            if node.val != preorder[-1]:
                node.left = makeTree()

            preorder.pop()
            return node

        return makeTree()

Yes. The confusing part is that `preorder[-1]` is **not telling us "does this node have a child?" directly**.

It's acting as a **completion marker**.

Forget the code for a moment. Let's understand why the two traversals can work together this way.

Take:

```text
        1
       / \
      2   3
     / \ / \
    4  5 6  7
```

Its traversals are:

```text
preorder  = [1,2,4,5,3,6,7]
postorder = [4,5,2,6,7,3,1]
```

This solution processes **both arrays from right to left**.

### Postorder backwards

Normal postorder:

```text
LEFT → RIGHT → ROOT
```

Backwards:

```text
ROOT → RIGHT → LEFT
```

So:

```text
postorder backwards:

1 → 3 → 7 → 6 → 2 → 5 → 4
```

This tells us **which node to create next**.

### Preorder backwards

Normal preorder:

```text
ROOT → LEFT → RIGHT
```

Backwards:

```text
RIGHT → LEFT → ROOT
```

So:

```text
preorder backwards:

7 → 6 → 3 → 5 → 4 → 2 → 1
```

This tells us **which node/subtree should finish next**.

That's the key relationship.

---

# Focus only on subtree 3

Ignore everything else.

```text
      3
     / \
    6   7
```

Its traversals are:

```text
preorder  = [3,6,7]
postorder = [6,7,3]
```

Reverse how we're consuming them:

```text
postorder backwards:
3 → 7 → 6

preorder backwards:
7 → 6 → 3
```

Notice the relationship:

```text
POST backwards:  3   7   6
                  ↑
             create root


PRE backwards:   7   6   3
                         ↑
                    finish root
```

Postorder backwards says:

> Start subtree `3`.

Preorder backwards says:

> You don't reach `3` until everything under `3` has been handled.

That is the intuition behind the comparison.

---

# Start `makeTree()` for 3

Suppose currently:

```text
postorder end → 3
preorder end  → 7
```

We do:

```python
node = TreeNode(postorder.pop())
```

So:

```text
node = 3
```

Now compare:

```python
node.val != preorder[-1]
```

which is:

```text
3 != 7
```

What does this tell us?

It tells us:

> We're at node 3, but the reverse-preorder completion pointer hasn't reached 3 yet.

There are still nodes belonging below `3` that need to be processed.

And because we're consuming postorder backwards:

```text
ROOT → RIGHT → LEFT
```

the first subtree we encounter is the **right subtree**.

So:

```python
node.right = makeTree()
```

---

# Now create 7

Next `postorder.pop()` gives:

```text
7
```

So:

```text
      3
       \
        7
```

At node 7:

```python
node.val != preorder[-1]
```

Current preorder end is also:

```text
7
```

So:

```text
7 != 7
```

is false.

This is the important moment.

The reverse-preorder completion pointer has already reached `7`.

There are no still-unprocessed nodes below `7` in this reconstruction.

So `7` is complete.

We don't recurse.

Then:

```python
preorder.pop()
```

removes `7`.

Now preorder ends with:

```text
6
```

Return node `7`.

---

# Back to node 3

This is why we need the **second `if`**.

Before building right:

```text
preorder[-1] = 7
```

But building `7` consumed it.

Now:

```text
preorder[-1] = 6
```

Our tree currently is:

```text
      3
       \
        7
```

Now ask again:

```python
if node.val != preorder[-1]:
```

That's:

```text
3 != 6
```

Still different.

Meaning:

> Even after finishing the right subtree, the completion pointer still hasn't reached `3`.

So there must still be another subtree to process before `3` is complete.

Because reverse postorder is:

```text
ROOT → RIGHT → LEFT
```

we've done RIGHT.

Next is LEFT.

So:

```python
node.left = makeTree()
```

---

# Create 6

Next postorder value:

```text
6
```

Create:

```text
      3
     / \
    6   7
```

At node 6:

```text
node.val = 6
preorder[-1] = 6
```

They match.

So `6` is complete.

Then:

```python
preorder.pop()
```

removes `6`.

Now preorder ends at:

```text
3
```

Return 6.

---

# Back to 3 again

Now:

```text
      3
     / \
    6   7
```

and:

```text
node.val = 3
preorder[-1] = 3
```

They match.

This means:

> Everything that needed to be processed before completing subtree 3 has now been consumed.

So `3` is complete.

Then:

```python
preorder.pop()
```

removes `3`.

Return the completed subtree.

---

# Why exactly does preorder work as the completion marker?

This is the deeper idea.

For subtree:

```text
      3
     / \
    6   7
```

Preorder is:

```text
3 6 7
```

But we're consuming preorder from the **end**:

```text
7 → 6 → 3
```

Notice:

**The root `3` is encountered last.**

That's true for every subtree when preorder is read backwards:

```text
RIGHT descendants
↓
LEFT descendants
↓
ROOT
```

Therefore, while building subtree `3`, as long as:

```python
preorder[-1] != 3
```

there are still descendant nodes that must finish before `3`.

When eventually:

```python
preorder[-1] == 3
```

all those descendants have been consumed.

So the subtree rooted at `3` is complete.

That's what the comparison really means.

---

# Why two `if`s instead of `while`?

Because a binary-tree node can have at most **two children**.

When processing reverse postorder, the order is:

```text
ROOT
RIGHT
LEFT
```

So for each node:

```python
if node not finished:
    build RIGHT

if node still not finished:
    build LEFT
```

After at most those two child subtrees, the current node must be complete.

Hence:

```python
if node.val != preorder[-1]:
    node.right = makeTree()

if node.val != preorder[-1]:
    node.left = makeTree()
```

The second condition must be checked **again** because building the right subtree changes `preorder[-1]`.

For node `3`:

```text
Start:
preorder[-1] = 7
3 != 7
→ build right

After right:
preorder[-1] = 6
3 != 6
→ build left

After left:
preorder[-1] = 3
→ 3 is complete
```

For a leaf `7`:

```text
Start:
preorder[-1] = 7

7 == 7
→ no right
→ no left
→ complete
```

And for a node where one child is sufficient to complete the subtree:

```text
Start:
node != preorder[-1]
→ build one child

After child:
node == preorder[-1]
→ don't build second child
```

Because preorder + postorder cannot distinguish whether a single child was originally left or right, this algorithm may place such a child on the right; the problem explicitly allows any valid reconstruction.

---

## Reduce the whole solution to two pointers

Think of the lists as stacks:

```text
POSTORDER stack
        ↓
"What node do I CREATE next?"

PREORDER stack
        ↓
"What node should FINISH next?"
```

So:

```python
node = TreeNode(postorder.pop())
```

means:

> Start constructing the next node/subtree.

And:

```python
node.val == preorder[-1]
```

means:

> The completion marker has reached this root, so its subtree is done.

And:

```python
preorder.pop()
```

means:

> Mark this root/subtree as finished and let its parent continue.

That's the invariant making this clever solution work.

For an interview, though, I would still prefer your **four-index + hashmap solution** unless you're fully comfortable proving this invariant. This version is shorter, but the boundary version is much easier to derive under pressure.


4. **https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/description/**

In [ ]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sortedArrayToBST(self, nums: List[int]) -> Optional[TreeNode]:
        def dfs(st,en):
            if st>en:
                return None
            mid = (st+en)//2
            root = TreeNode(nums[mid])
            root.left = dfs(st,mid-1)
            root.right = dfs(mid+1,en)
            return root
        return dfs(0,len(nums)-1)

5. **https://leetcode.com/problems/convert-sorted-list-to-binary-search-tree/description/**

In [ ]:
# time = O(n)
# space = O(n), using etc arr for storing nums

# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sortedListToBST(self, head: Optional[ListNode]) -> Optional[TreeNode]:
        arr = []
        curr = head
        while curr:
            arr.append(curr.val)
            curr = curr.next
        
        def dfs(st,end):
            if st>end:
                return None
            mid = (st+end)//2
            root = TreeNode(arr[mid])
            root.left = dfs(st,mid-1)
            root.right = dfs(mid+1,end)
            return root
        return dfs(0,len(arr)-1)

In [ ]:
Array solution       → O(n) time, O(n) extra space
Slow/fast solution   → O(n log n) time, O(log n) stack

In [ ]:
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def getMiddleAndSplit(self,head):
        slow = head
        prev = slow
        fast = head
        while fast and fast.next:
            prev = slow
            slow = slow.next
            fast = fast.next.next
        if prev:
            prev.next = None
        return slow
    def sortedListToBST(self, head: Optional[ListNode]) -> Optional[TreeNode]:
        if(head is None):
            return None
        if(head.next is None):
            return TreeNode(head.val)
        middle = self.getMiddleAndSplit(head)
        root = TreeNode(middle.val)
        root.left = self.sortedListToBST(head)
        root.right = self.sortedListToBST(middle.next)
        return root